# Customer Churn Hive ETL
Notebook này dùng để kết nối Hive, chạy DDL/DML cho tầng raw và curated, và kiểm tra output churn analytics.

## 1. Cài thư viện trong notebook
Chạy cell cài package trước khi import. Bạn có thể chỉnh package theo cách kết nối Hive bạn chọn.

In [78]:
# Uncomment when ready to install libraries in the notebook ke rnel
%pip install pyhive thrift thrift-sasl pandas sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 13.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.5/613.5 kB 11.6 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Cấu hình kết nối Hive

In [188]:
from sqlalchemy import create_engine, text
import pandas as pd

class HiveClient:
    def __init__(self, connection_string = ''):
        self.engine = create_engine(connection_string)
        print(self.engine)
        
    def get_engine(self):
        return self.engine
    
    def execute(self, query: str):
        with self.engine.connect() as conn:
            conn.execute(text(query))      
            
    def query(self, query: str) -> pd.DataFrame:
        with self.engine.connect() as conn:
            conn.execute(text("SET hive.execution.engine=mr"))
            # conn.execute(text("SET hive.exec.dynamic.partition=true"))
            # conn.execute(text("SET hive.exec.dynamic.partition.mode=nonstrict"))
            # conn.execute(text("SET hive.auto.convert.join=false"))
            # conn.execute(text("SET hive.exec.max.dynamic.partitions=100000"))
            # conn.execute(text("SET hive.exec.max.dynamic.partitions.pernode=10000"))
            conn.execute(text("SET hive.stats.autogather=false"))           # ← most important
            conn.execute(text("SET hive.cbo.enable=false"))                 # disable cost-based optimizer
            conn.execute(text("SET hive.stats.column.autogather=false"))
            conn.execute(text("SET hive.vectorized.execution.enabled=false"))
            return pd.read_sql(query, conn)

In [189]:
hive_client = HiveClient('hive://root@dtwarehouse-hiveserver2:10000')

Engine(hive://root@dtwarehouse-hiveserver2:10000)


In [190]:
raw_ddl = '''CREATE DATABASE IF NOT EXISTS raw_churn'''
create_table_raw_churn_customers_raw = '''
CREATE EXTERNAL TABLE IF NOT EXISTS raw_churn.customers_raw (
  customer_id STRING,
  signup_date DATE,
  birth_year INT,
  gender STRING,
  city STRING,
  acquisition_channel STRING,
  segment STRING,
  is_active BOOLEAN
)
PARTITIONED BY (dt STRING)
STORED AS PARQUET
LOCATION 'hdfs://namenode:8020/data/raw/churn/customers'
'''
create_table_raw_churn_orders_raw = '''
CREATE EXTERNAL TABLE IF NOT EXISTS raw_churn.orders_raw (
  order_id STRING,
  customer_id STRING,
  order_ts TIMESTAMP,
  order_status STRING,
  currency STRING,
  subtotal_amount DECIMAL(18,2),
  discount_amount DECIMAL(18,2),
  shipping_fee DECIMAL(18,2),
  tax_amount DECIMAL(18,2),
  total_amount DECIMAL(18,2),
  payment_method STRING,
  promo_code STRING
)
PARTITIONED BY (dt STRING)
STORED AS PARQUET
LOCATION 'hdfs://namenode:8020/data/raw/churn/orders'
'''
daily_tables_sql = '''
CREATE TABLE IF NOT EXISTS raw_churn.customer_features_daily (
  customer_id STRING,

  -- profile
  signup_date DATE,
  birth_year INT,
  gender STRING,
  city STRING,
  acquisition_channel STRING,
  segment STRING,
  is_active BOOLEAN,

  -- daily metrics
  orders_count INT,
  total_spent DECIMAL(18,2),
  last_order_ts TIMESTAMP,

  -- rolling features
  days_since_last_order INT,
  orders_30d INT

)
PARTITIONED BY (dt STRING)
STORED AS PARQUET
'''

risk_tier_sql = '''
WITH latest_dt AS (
  SELECT MAX(dt) AS dt
  FROM raw_churn.customer_features_daily
),

base AS (
  SELECT
    f.customer_id,
    f.dt,

    CASE
      WHEN COALESCE(f.days_since_last_order, 0) >= 60 THEN 0.85
      WHEN COALESCE(f.days_since_last_order, 0) >= 45 THEN 0.75
      WHEN COALESCE(f.days_since_last_order, 0) >= 30 THEN 0.55
      WHEN COALESCE(f.orders_30d, 0) = 0 THEN 0.50
      ELSE 0.20
    END AS churn_risk_score

  FROM raw_churn.customer_features_daily f
  JOIN latest_dt d
    ON f.dt = d.dt
)

SELECT
  customer_id,
  dt,
  churn_risk_score,

  CASE
    WHEN churn_risk_score >= 0.8 THEN 'High risk'
    WHEN churn_risk_score >= 0.5 THEN 'Medium risk'
    ELSE 'Low risk'
  END AS risk_tier

FROM base
ORDER BY churn_risk_score DESC, customer_id
LIMIT 100
'''

In [191]:
hive_client.execute(raw_ddl)
hive_client.execute(create_table_raw_churn_customers_raw)
hive_client.execute(daily_tables_sql)
hive_client.execute(create_table_raw_churn_orders_raw)


In [192]:
customer_fs = pd.read_sql(
    "SELECT * FROM raw_churn.customers_raw LIMIT 100",
    hive_client.get_engine()
)
customer_fs

,customer_id,signup_date,birth_year,gender,city,acquisition_channel,segment,is_active,dt
0,CUST_000001,2026-03-16,1985.0,female,Ha Noi,ads,regular,True,2026-04-11
1,CUST_000002,2026-01-11,1972.0,male,Ho Chi Minh,organic,vip,True,2026-04-11
2,CUST_000003,2024-09-13,2004.0,unknown,Ha Noi,social,regular,True,2026-04-11
3,CUST_000004,2026-04-04,1980.0,unknown,Da Nang,referral,regular,True,2026-04-11
4,CUST_000005,2025-05-01,1994.0,male,Da Nang,referral,vip,True,2026-04-11
...,...,...,...,...,...,...,...,...,...
95,CUST_000096,2025-06-16,NaN,other,Nha Trang,referral,regular,True,2026-04-11
96,CUST_000097,2024-05-07,1986.0,female,Can Tho,ads,regular,True,2026-04-11
97,CUST_000098,2025-07-12,1974.0,male,Ha Noi,referral,regular,True,2026-04-11
98,CUST_000099,2025-06-19,1999.0,other,Nha Trang,social,regular,False,2026-04-11


In [193]:
order_df = pd.read_sql(
    "SELECT * FROM raw_churn.orders_raw LIMIT 100",
    hive_client.get_engine()
)
order_df

,order_id,customer_id,order_ts,order_status,currency,subtotal_amount,discount_amount,shipping_fee,tax_amount,total_amount,payment_method,promo_code,dt
0,ORD_20251212_00000003,CUST_000648,2025-12-12 18:45:17.924192,completed,VND,158974.0,0.00,15000.0,12717.92,186691.92,ewallet,NaN,2025-12-12
1,ORD_20251212_00000127,CUST_000432,2025-12-12 09:36:41.924192,completed,VND,246817.0,0.00,25000.0,19745.36,291562.36,bank_transfer,VIP30,2025-12-12
2,ORD_20251212_00000437,CUST_000166,2025-12-12 13:08:32.924192,completed,VND,570282.0,0.00,25000.0,45622.56,640904.56,bank_transfer,NaN,2025-12-12
3,ORD_20251212_00000442,CUST_000508,2025-12-12 16:37:47.924192,completed,VND,985045.0,0.00,15000.0,78803.60,1078848.60,bank_transfer,NaN,2025-12-12
4,ORD_20251212_00000663,CUST_000250,2025-12-12 23:26:30.924192,completed,VND,1161987.0,0.00,25000.0,92958.96,1279945.96,card,VIP30,2025-12-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,ORD_20251214_00002545,CUST_000086,2025-12-14 20:40:30.924192,completed,VND,366929.0,55039.35,25000.0,24951.17,361840.82,cod,NaN,2025-12-14
96,ORD_20251214_00002574,CUST_000741,2025-12-14 13:07:53.924192,completed,VND,550739.0,55073.90,0.0,39653.21,535318.31,card,LOYAL20,2025-12-14
97,ORD_20251214_00002899,CUST_000897,2025-12-14 00:30:51.924192,completed,VND,264384.0,0.00,0.0,21150.72,285534.72,ewallet,LOYAL20,2025-12-14
98,ORD_20251214_00002946,CUST_000117,2025-12-14 16:45:42.924192,completed,VND,152967.0,0.00,15000.0,12237.36,180204.36,bank_transfer,NaN,2025-12-14


In [194]:
df = hive_client.query(risk_tier_sql)
df

DatabaseError: Execution failed on sql '
WITH latest_dt AS (
  SELECT MAX(dt) AS dt
  FROM raw_churn.customer_features_daily
),

base AS (
  SELECT
    f.customer_id,
    f.dt,

    CASE
      WHEN COALESCE(f.days_since_last_order, 0) >= 60 THEN 0.85
      WHEN COALESCE(f.days_since_last_order, 0) >= 45 THEN 0.75
      WHEN COALESCE(f.days_since_last_order, 0) >= 30 THEN 0.55
      WHEN COALESCE(f.orders_30d, 0) = 0 THEN 0.50
      ELSE 0.20
    END AS churn_risk_score

  FROM raw_churn.customer_features_daily f
  JOIN latest_dt d
    ON f.dt = d.dt
)

SELECT
  customer_id,
  dt,
  churn_risk_score,

  CASE
    WHEN churn_risk_score >= 0.8 THEN 'High risk'
    WHEN churn_risk_score >= 0.5 THEN 'Medium risk'
    ELSE 'Low risk'
  END AS risk_tier

FROM base
ORDER BY churn_risk_score DESC, customer_id
LIMIT 100
': (pyhive.exc.OperationalError) TExecuteStatementResp(status=TStatus(statusCode=3, infoMessages=['Server-side error; please check HS2 logs.'], sqlState='08S01', errorCode=2, erro